In [3]:
import pandas as pd
import numpy as np
import re
import nltk
import spacy
import warnings 
warnings.filterwarnings('ignore')

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
from nltk.corpus import stopwords

nlp = spacy.load('en_core_web_sm')

STOP = set(stopwords.words('english'))

print('Libraries imported and resources downloaded successfully!')
print(f'stopwords count: {len(STOP)}')

Libraries imported and resources downloaded successfully!
stopwords count: 198


In [6]:
df = pd.read_csv('../../data/processed/youtoxic_clean.csv')

print(f'Shape of dataset: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print(f'\nClass distribution:')
print(df['IsToxic'].value_counts())
df.head(5)

Shape of dataset: (995, 2)
Columns: ['Text', 'IsToxic']

Class distribution:
IsToxic
False    538
True     457
Name: count, dtype: int64


,Text,IsToxic
0,If only people would just take a step back and...,False
1,Law enforcement is not trained to shoot to app...,True
2,\nDont you reckon them 'black lives matter' ba...,True
3,There are a very large number of people who do...,False
4,"The Arab dude is absolutely right, he should h...",False


## 2. Preprocessing Pipeline

### Decision log
| Step | Tool | Why |
|---|---|---|
| Remove URLs, mentions, HTML | regex | No toxicity signal |
| Normalize repeated chars | regex | loooool → lool |
| Remove punctuation and numbers | regex | Reduce vocabulary noise |
| Lowercase | Python | Standardize |
| Remove stopwords | NLTK | More complete than SpaCy |
| Lemmatization | SpaCy | Returns real base forms, cleaner than stemming |
| Filter tokens < 3 chars | Python | Remove noise |

### Why lemmatization over stemming
- Stemming: hating → hat (not a real word)
- Lemmatization: hating → hate (real base form)

### Why NLTK stopwords + SpaCy lemmatization
- NLTK stopwords list is more complete (179 words vs SpaCy)
- SpaCy is better for lemmatization
- Use each tool for what it does best

In [8]:
def clean_text(text: str) -> str:
    """Remove noise from raw text.
    
    Steps:
        1. Convert to string (safety)
        2. Remove URLs
        3. Remove mentions (@user)
        4. Remove HTML tags
        5. Normalize repeated chars (loooool → lool)
        6. Remove punctuation and numbers
        7. Remove extra whitespace
        8. Lowercase and strip
    
    Args:
        text: Raw comment string
    
    Returns:
        Cleaned lowercase string
    """

    text = str(text)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)     # URLs
    text = re.sub(r'@w+', ' ', text)                       # Mentions
    text = re.sub(r'<[^>]+>', ' ', text)                   # HTML tags
    text = re.sub(r'(.)\1{2,}', r'\1', text)               # Repeated chars
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)               # Punctuation & numbers
    text = re.sub(r'\s+', ' ', text)                       # multiple spaces → single space
    text = text.lower().strip()
    return text

# Test the function
test_cases = [
    'I HATE you!!! You are the WORST 😡😡',
    'Check this out: https://youtube.com @user123',
    '<b>loooool</b> wtf!!!! is this???',
    'Run them over!!!',
]

print('Original → Cleaned')
for text in test_cases:
    print(f'input  : {text}')
    print(f'output : {clean_text(text)}\n')

Original → Cleaned
input  : I HATE you!!! You are the WORST 😡😡
output : i hate you you are the worst

input  : Check this out: https://youtube.com @user123
output : check this out user

input  : <b>loooool</b> wtf!!!! is this???
output : lol wtf is this

input  : Run them over!!!
output : run them over



In [9]:
def lemmatize_text(text: str) -> str:
    """Lemmatize text using SpaCy and remove stopwords with NLTK.
    
    Steps:
        1. Parse text with SpaCy
        2. Get lemma for each token
        3. Filter: remove stopwords (NLTK), spaces, tokens < 3 chars
        4. Join tokens back to string
    
    Args:
        text: Cleaned lowercase string
    
    Returns:
        Lemmatized string without stopwords
    """
    doc = nlp(text)
    tokens = [
        token.lemma_ 
        for token in doc
        if token.lemma_ not in STOP 
        and not token.is_space
        and len(token.lemma_) > 2
    ]

    return ' '.join(tokens)

# Test the function
test_cases = [
    'i hate you so much you are the worst person ever',
    'the police officers were running after the criminals',
    'fucking idiot stop being so racist all the time',
]

print('Cleaned → Lemmatized')
for text in test_cases:
    print(f'input  : {text}')
    print(f'output : {lemmatize_text(text)}\n')

Cleaned → Lemmatized
input  : i hate you so much you are the worst person ever
output : hate much bad person ever

input  : the police officers were running after the criminals
output : police officer run criminal

input  : fucking idiot stop being so racist all the time
output : fucking idiot stop racist time



In [11]:
def preprocess_text(text: str) -> str:
    """Full preprocessing pipeline: clean + lemmatize.
    
    Args:
        text: Raw comment string
    
    Returns:
        Fully preprocessed string
    """
    cleaned = clean_text(text)
    lemmatized = lemmatize_text(cleaned)
    return lemmatized

# Test the full pipeline
test_cases = [
    'I HATE you!!! You are the WORST person ever 😡😡',
    'Check https://youtube.com @user123 this video is amazing!!!',
    'You fucking idiot, go back to where you came from!!!',
    'wtf',
    'Vile cunt',
]

print("==Full Preprocessing Pipeline==")
for text in test_cases:
    print(f'raw   : {text}')
    print(f'Clean : {clean_text(text)}')
    print(f'Final : {preprocess_text(text)}\n')

==Full Preprocessing Pipeline==
raw   : I HATE you!!! You are the WORST person ever 😡😡
Clean : i hate you you are the worst person ever
Final : hate bad person ever

raw   : Check https://youtube.com @user123 this video is amazing!!!
Clean : check user this video is amazing
Final : check user video amazing

raw   : You fucking idiot, go back to where you came from!!!
Clean : you fucking idiot go back to where you came from
Final : fucking idiot back come

raw   : wtf
Clean : wtf
Final : wtf

raw   : Vile cunt
Clean : vile cunt
Final : vile cunt



In [13]:
print('Applying preprocessing pipeline...')
print(f'input rows: {len(df)}')

df['text_preprocessed'] = df['Text'].apply(preprocess_text)

print(f'Output rows: {len(df)}')
print('Done')

Applying preprocessing pipeline...
input rows: 995
Output rows: 995
Done


In [15]:
empty = df[df['text_preprocessed'].str.strip() == '']
print(f'Empty text after preprocessing: {len(empty)}')

if len(empty) > 0:
    print('Empty text found after preprocessing.')
    print(empty[['Text', 'text_preprocessed', 'IsToxic']])

df['prep_word_count'] = df['text_preprocessed'].str.split().str.len()
short = df[df['prep_word_count'] <= 1]
print(f'\nTexts with <= 1 word after preprocessing: {len(short)}')
if len(short) > 0:
    print(short[['Text', 'text_preprocessed', 'IsToxic']].to_string())

df['raw_word_count'] = df['Text'].fillna('').str.split().str.len()
print('\n== Word count comparison ==')
print((f'Raw median words         : {df["raw_word_count"].median():.0f}' ))
print((f'Preprocessed median words: {df["prep_word_count"].median():.0f}'))
print((f'Vocabulary reduction     : {(1 - df["prep_word_count"].median() / df["raw_word_count"].median()) * 100:.0f}%'))

df = df[df['prep_word_count'] > 1].reset_index(drop=True)
print(f'\nDataset after removing short texts: {df.shape}')


Empty text after preprocessing: 1
Empty text found after preprocessing.
                            Text text_preprocessed  IsToxic
896  7:24 7:30 XDDDDDDDDDDDDDDDD                      False

Texts with <= 1 word after preprocessing: 33
                                Text     text_preprocessed  IsToxic
77             Iz didn't do nuffinz!               nuffinz    False
105                           Wow!!!                   wow    False
111                      Well done 👍                  well    False
225            Was this in Virginia?              virginia    False
231                 should have shot                 shoot     True
249                   Ah, beautiful.             beautiful     True
303                        on drugs*                  drug    False
305                  Its all a hoax                   hoax    False
342                     Nicely done.                nicely    False
353                     @12:25 LMFAO                 lmfao    False
433           

In [16]:
print('=== Before vs after preprocessing: ===')
print()

sample = df.sample(8, random_state=42)
for _, row in sample.iterrows():
    label = 'Toxic' if row['IsToxic'] == 1 else 'Non-toxic'
    print(f'Label: {label}')
    print(f' raw:  {str(row["Text"])[:100]}')
    print(f' Final: {row["text_preprocessed"][:100]}')
    print()

=== Before vs after preprocessing: ===

Label: Non-toxic
 raw:  You're the man, Stefan!
 Final: man stefan

Label: Non-toxic
 raw:  12:28 - can someone please turn that into an auto-tune rap song by Stefan Molyneux? Please.
 Final: someone please turn auto tune rap song stefan molyneux please

Label: Toxic
 raw:  OMG Stefan that rap was so hysterical. anyway, good news I guess, to know that whatshisname is douch
 Final: omg stefan rap hysterical anyway good news guess know whatshisname douche deserve die premptive stri

Label: Non-toxic
 raw:  Wake up Black People.
 Final: wake black people

Label: Non-toxic
 raw:  you published this on Aug 17 , the "truth'' is still to be uncovered, we know what is likely,You can
 Final: publish aug truth still uncover know likely assume know entire story investigate fed investigate bac

Label: Toxic
 raw:  If this happen I would drive though them if I have to n if a cop does that Shit then they mess up le
 Final: happen would drive though cop shit me

In [ ]:
preprocessing_module = '''
"""Text preprocessing pipeline for hate speech classification."""

import re
import nltk
import spacy

nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

nlp = spacy.load('en_core_web_sm')
STOP = set(stopwords.words('english'))


def clean_text(text: str) -> str:
    """Remove noise from raw text."""
    text = str(text)
    text = re.sub(r'https?://\\S+|www\\.\\S+', ' ', text)
    text = re.sub(r'@\\w+', ' ', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'(.)\\1{2,}', r'\\1\\1', text)
    text = re.sub(r'[^a-zA-Z\\s]', ' ', text)
    text = re.sub(r'\\s+', ' ', text)
    text = text.lower().strip()
    return text


def lemmatize_text(text: str) -> str:
    """Lemmatize text using SpaCy and remove stopwords with NLTK."""
    doc = nlp(text)
    tokens = [
        token.lemma_
        for token in doc
        if token.lemma_ not in STOP
        and not token.is_space
        and len(token.lemma_) > 2
    ]
    return ' '.join(tokens)


def preprocess(text: str) -> str:
    """Full preprocessing pipeline: clean + lemmatize."""
    return lemmatize_text(clean_text(text))
'''

with open('../../src/features/preprocessing.py', 'w') as f:
    f.write(preprocessing_module)

print('Saved: src/features/preprocessing.py')



✅ Saved: src/features/preprocessing.py


In [19]:
preprocessing_module = '''
"""Text preprocessing pipeline for hate speech classification."""

import re
import nltk
import spacy

nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

nlp = spacy.load('en_core_web_sm')
STOP = set(stopwords.words('english'))


def clean_text(text: str) -> str:
    """Remove noise from raw text."""
    text = str(text)
    text = re.sub(r'https?://\\S+|www\\.\\S+', ' ', text)
    text = re.sub(r'@\\w+', ' ', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'(.)\\1{2,}', r'\\1\\1', text)
    text = re.sub(r'[^a-zA-Z\\s]', ' ', text)
    text = re.sub(r'\\s+', ' ', text)
    text = text.lower().strip()
    return text


def lemmatize_text(text: str) -> str:
    """Lemmatize text using SpaCy and remove stopwords with NLTK."""
    doc = nlp(text)
    tokens = [
        token.lemma_
        for token in doc
        if token.lemma_ not in STOP
        and not token.is_space
        and len(token.lemma_) > 2
    ]
    return ' '.join(tokens)


def preprocess(text: str) -> str:
    """Full preprocessing pipeline: clean + lemmatize."""
    return lemmatize_text(clean_text(text))
'''

with open('../../src/features/preprocessing.py', 'w') as f:
    f.write(preprocessing_module)

print('Saved: src/features/preprocessing.py')

Saved: src/features/preprocessing.py


In [21]:
conclusions = {
    'Input rows':           '995 rows from youtoxic_clean.csv',
    'Output rows':          '962 rows — 33 short texts + 1 empty removed',
    'Cleaning steps':       'URLs, mentions, HTML, repeated chars, punctuation, numbers',
    'Lemmatization':        'SpaCy en_core_web_sm — real base forms',
    'Stopwords':            'NLTK — more complete list (179 words)',
    'Token filter':         'Tokens < 3 chars removed',
    'Vocabulary reduction': '~47% fewer words after preprocessing',
    'Empty texts':          '1 empty text found and removed',
    'Class balance':        f"Not Toxic: {(df['IsToxic']==False).sum()} | Toxic: {(df['IsToxic']==True).sum()}",  # ← AQUÍ
    'Pipeline location':    'src/features/preprocessing.py — reusable module',
    'Next step':            'Vectorization (TF-IDF) + ML model training',
}
print('=== PREPROCESSING CONCLUSIONS ===')
for k, v in conclusions.items():
    print(f'  {k:25s} → {v}')

=== PREPROCESSING CONCLUSIONS ===
  Input rows                → 995 rows from youtoxic_clean.csv
  Output rows               → 962 rows — 33 short texts + 1 empty removed
  Cleaning steps            → URLs, mentions, HTML, repeated chars, punctuation, numbers
  Lemmatization             → SpaCy en_core_web_sm — real base forms
  Stopwords                 → NLTK — more complete list (179 words)
  Token filter              → Tokens < 3 chars removed
  Vocabulary reduction      → ~47% fewer words after preprocessing
  Empty texts               → 1 empty text found and removed
  Class balance             → Not Toxic: 516 | Toxic: 446
  Pipeline location         → src/features/preprocessing.py — reusable module
  Next step                 → Vectorization (TF-IDF) + ML model training


In [25]:
df_out = df[['text_preprocessed', 'IsToxic']].copy()
df_out.columns = ['Text', 'IsToxic']

df_out.to_csv('../../data/processed/youtoxic_preprocessed.csv', index=False)

print('Saved: data/processed/youtoxic_preprocessed.csv')
print(f'Shape of final dataset: {df_out.shape}')
print(f'Columns               : {df_out.columns.tolist()}')
print('\n Class distribution:')
print(df_out['IsToxic'].value_counts())
print(f'\n Sample preprocessed texts:')
for _, row in df_out.sample(5, random_state=42).iterrows():
    label = 'Toxic' if row['IsToxic'] else 'Non-toxic'
    print(f'Label: {label}    | Text: {row["Text"][:80]}')

Saved: data/processed/youtoxic_preprocessed.csv
Shape of final dataset: (962, 2)
Columns               : ['Text', 'IsToxic']

 Class distribution:
IsToxic
False    516
True     446
Name: count, dtype: int64

 Sample preprocessed texts:
Label: Non-toxic    | Text: man stefan
Label: Non-toxic    | Text: someone please turn auto tune rap song stefan molyneux please
Label: Toxic    | Text: omg stefan rap hysterical anyway good news guess know whatshisname douche deserv
Label: Non-toxic    | Text: wake black people
Label: Non-toxic    | Text: publish aug truth still uncover know likely assume know entire story investigate
